# Setup de almacenamiento

Este notebook configura los objetos de Unity Catalog necesarios
para acceder al almacenamiento externo de DEV o PROD.

Recursos:
- Storage Credential
- External Locations

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.dropdown("environment","dev",["dev", "prod"],"Environment")

environment = dbutils.widgets.get("environment")

In [0]:
environments = {
    "dev": {
        "storage_account": "storagetoxdev",
        "storage_credential": "storage_credential_dev",
        "access_connector_id": (
            "/subscriptions/a4e60f2c-c6d9-45f2-b90b-614f8504fbd3/resourceGroups/resource_toxqui/providers/Microsoft.Databricks/accessConnectors/connector_dev"
            "/resourceGroups/resource_toxqui"
            "/providers/Microsoft.Databricks/accessConnectors/connector_dev"
        )
    },

    "prod": {
        "storage_account": "storagetoxprod",
        "storage_credential": "storage_credential_prod",
        "access_connector_id": (
            "/subscriptions/a4e60f2c-c6d9-45f2-b90b-614f8504fbd3/resourceGroups/resource_toxqui/providers/Microsoft.Databricks/accessConnectors/connector_prod"
            "/resourceGroups/resource_toxqui"
            "/providers/Microsoft.Databricks/accessConnectors/connector_prod"
        )
    }
}

In [0]:
config = environments[environment]

storage_account = config["storage_account"]
credential_name = config["storage_credential"]
access_connector_id = config["access_connector_id"]

containers = [
    "raw",
    "bronze",
    "silver",
    "gold",
    "backup"
]

print(f"Environment: {environment}")
print(f"Storage: {storage_account}")
print(f"Credential: {credential_name}")

#Crear Storage Credential

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog

w = WorkspaceClient()

existing_credentials = {
    credential.name
    for credential in w.storage_credentials.list()
}

if credential_name in existing_credentials:
    print(f"Storage Credential ya existe: {credential_name}")
else:
    w.storage_credentials.create(
        name=credential_name,
        azure_managed_identity=catalog.AzureManagedIdentityRequest(
            access_connector_id=access_connector_id
        ),
        read_only=False,
        comment=f"Storage credential para {environment}"
    )

    print(f"Storage Credential creada: {credential_name}")

In [0]:
for container in containers:

    external_location = f"ext_{environment}_{container}"

    storage_url = (
        f"abfss://{container}@"
        f"{storage_account}.dfs.core.windows.net/"
    )

    sql = f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS `{external_location}`
    URL '{storage_url}'
    WITH (STORAGE CREDENTIAL `{credential_name}`)
    COMMENT 'External location {container} - {environment}'
    """

    spark.sql(sql)

    print(f"OK: {external_location}")

In [0]:
for location in w.external_locations.list():
    if location.name.startswith(f"ext_{environment}_"):
        print(f"{location.name} -> {location.url}")

##Isolacion de las credenciales 


workspace_id = w.get_workspace_id()

print(f"Workspace ID actual: {workspace_id}")

from databricks.sdk.service import catalog

for container in containers:
    external_location = f"ext_{environment}_{container}"

    w.external_locations.update(
        name=external_location,
        isolation_mode=catalog.IsolationMode.ISOLATION_MODE_ISOLATED
    )

    w.workspace_bindings.update_bindings(
        securable_type="external_location",
        securable_name=external_location,
        add=[
            catalog.WorkspaceBinding(
                workspace_id=workspace_id
            )
        ]
    )

    print(f"OK: {external_location} -> workspace {workspace_id}")

In [0]:
catalog_name = f"smart_data_{environment}"
schema_name = "storage"

managed_location = (
    f"abfss://gold@{storage_account}.dfs.core.windows.net/_managed/{catalog_name}"
)

print(f"Catalog: {catalog_name}")
print(f"Schema : {schema_name}")
print(f"Managed location: {managed_location}")

##creara catalogo y schema

In [0]:
spark.sql(f"""
CREATE CATALOG IF NOT EXISTS `{catalog_name}`
MANAGED LOCATION '{managed_location}'
COMMENT 'Catalogo del proyecto smart_data - ambiente {environment}'
""")

print(f"OK: catalogo {catalog_name}")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`
COMMENT 'Volumes de almacenamiento del proyecto'
""")

print(f"OK: schema {catalog_name}.{schema_name}")

Creamos volumenes

In [0]:
for container in containers:

    volume_name = container

    volume_location = (
        f"abfss://{container}@"
        f"{storage_account}.dfs.core.windows.net/files"
    )

    spark.sql(f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS
        `{catalog_name}`.`{schema_name}`.`{volume_name}`
    LOCATION '{volume_location}'
    COMMENT 'Volume {container} - ambiente {environment}'
    """)

    print(
        f"OK: {catalog_name}.{schema_name}.{volume_name}"
        f" -> {volume_location}"
    )

In [0]:
display(
    spark.sql(
        f"SHOW VOLUMES IN `{catalog_name}`.`{schema_name}`"
    )
)

In [0]:
test_file = (
    f"/Volumes/{catalog_name}/{schema_name}/raw/"
    "test_connection.csv"
)

print(test_file)

In [0]:
with open(test_file, "w") as f:
    f.write("id,name\n")
    f.write("1,Databricks\n")
    f.write("2,Azure\n")

print("OK: archivo creado")

In [0]:
import os

if os.path.exists(test_file):
    os.remove(test_file)

print("OK: archivo de prueba eliminado")